# 08 · Neighbors, Leiden clusters and UMAP

*sc-rnaseq-wf, re-instantiated step by step for `team-jakobsson` (`jakobsson-pmdbs-sn-rnaseq`)*

| | |
| :- | :- |
| WDL task | [`cluster_data.cluster_cells`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/workflows/cohort_analysis/cluster_data/cluster_data.wdl) |
| Script | [`clustering_umap`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/docker/sc_tools/scripts/main/clustering_umap) → `sc-clustering-umap` |

On the scVI latent space (`X_scVI`):

1. a k-nearest-neighbour graph with `n_neighbors = 15` (CAGRA approximate nearest neighbours),
2. Leiden clustering at each resolution in `leiden_res = [0.2, 0.5, 1.0]` (`n_iterations=2`), stored as `leiden_res_0.20`, `leiden_res_0.50`, `leiden_res_1.00`,
3. a UMAP embedding (`X_umap`) for plotting.

All three run on the GPU with [`rapids-singlecell`](https://github.com/scverse/rapids-singlecell). **Requires a GPU.**

The workflow clusters after scANVI labeling. Clustering only uses the scVI latent space, so running it before `sc-annotate` (notebook 09) gives the same result and lets the final object carry the UMAP.

**Inputs**
- `${cohort_id}.scvi_integrated.h5ad`

**Outputs** (in `cfg.RUN_DIR`, named as in the WDL)
- `${cohort_id}.umap_clustered.h5ad` (adds `obs['leiden_res_*']`, `obsm['X_umap']`)

**Previous:** [07_integrate_scvi](07_integrate_scvi.ipynb) · **Next:** [09_annotate](09_annotate.ipynb)

In [ ]:
import wf_config as cfg
import pandas as pd
import scanpy as sc
from IPython.display import Image, display

cfg.RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run directory:", cfg.RUN_DIR)

## 1. Run `sc-clustering-umap`

In [ ]:
cfg.run(
    "sc-clustering-umap",
    "--latent-key", cfg.SCVI_LATENT_KEY,
    "--n-neighbors", cfg.N_NEIGHBORS,
    "--leiden-res", *cfg.LEIDEN_RES,
    "--adata-input", cfg.out("scvi_integrated.h5ad"),
    "--adata-output", cfg.out("umap_clustered.h5ad"),
)

## 2. Clusters and UMAP

In [ ]:
adata = sc.read_h5ad(cfg.out("umap_clustered.h5ad"))
leiden_keys = [f"leiden_res_{r:4.2f}" for r in cfg.LEIDEN_RES]
print({k: adata.obs[k].nunique() for k in leiden_keys})

In [ ]:
sc.pl.umap(adata, color=leiden_keys + ["sample", "batch_id"], ncols=2, frameon=False)

If cells group by `sample` or `batch_id` rather than forming shared clusters, integration has not removed the batch effect; the scib metrics in notebook 10 measure this.

**Previous:** [07_integrate_scvi](07_integrate_scvi.ipynb) · **Next:** [09_annotate](09_annotate.ipynb)